Data Science Fundamentals for DCBP, University of Bern

# CodingTask 2: Polymer Feature Hunt

- Submission format: notebook only (`.ipynb`)
- Work style: individual or small teams of up to 3 allowed
- Focus: **chemistry-driven feature engineering** and interpretation of model failures
- Prerequisites: artifacts from CodingTask 1 (`artifacts/ffv_features_rdkit.csv`) -- the notebook below rebuilds it from `data/train.csv` if missing

- **Deadline**: 2026-05-17, 23:59 (send your notebook by email to matteo.boi@unibe.ch)
- **On-site review/experiment lecture**: 2026-05-20
- Estimated effort: 4-6 hours

---

## What this task is about

1. **Why** you picked the chemical features you picked
2. **What** your model gets wrong, on **which** polymers, and **why** (interpretation of failure cases)

So spend most of your effort on the *reasoning*, not on the *code*.

## Points (12.5 total + up to 1 bonus)

| Component | Pts | What earns full credit |
|---|---|---|
| CT2.1 Feature picking | 6.0 | At least 5 new features, each with a one-sentence justification |
| CT2.2 Outlier interpretation | 4.0 | 5 worst polymers identified, structures drawn, **structural hypothesis** is specific (functional groups, ring systems, chain length, etc.) |
| Participation in May 20 session | 2.5 | Asks at least one question OR proposes a feature OR contributes to discussion |
| **Bonus** -- alt model / sweep | +1.0 | Reasonable alternative model attempted with brief discussion |


In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

---
## Loading the dataset

The task uses the same dataset and split as Notebook 13. The cell below rebuilds `artifacts/ffv_features_rdkit.csv` from `data/train.csv` if it is missing, so you can run this notebook even if you skipped some of CodingTask 1.

In [6]:
rdkit_path = ARTIFACT_DIR / "ffv_features_rdkit.csv"

if not rdkit_path.exists():
    print("Rebuilding artifacts/ffv_features_rdkit.csv from data/train.csv ...")
    train_df = pd.read_csv("data/train.csv")
    ffv = train_df.dropna(subset=["FFV"])[["id", "SMILES", "FFV"]].copy()

    # Basic SMILES features (same as CodingTask 1)
    ffv["smiles_len"]  = ffv["SMILES"].str.len()
    ffv["count_star"]  = ffv["SMILES"].str.count(r"\*")
    ffv["count_C"]     = ffv["SMILES"].str.count("C")
    ffv["count_O"]     = ffv["SMILES"].str.count("O")
    ffv["count_N"]     = ffv["SMILES"].str.count("N")
    ffv["count_equal"] = ffv["SMILES"].str.count("=")

    # Three baseline RDKit descriptors
    def rdkit_row(smi):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return [np.nan, np.nan, np.nan]
        return [Descriptors.MolWt(mol),
                Descriptors.NumValenceElectrons(mol),
                Descriptors.TPSA(mol)]

    rdkit_vals = np.array([rdkit_row(s) for s in ffv["SMILES"]])
    ffv["MolWt"]                 = rdkit_vals[:, 0]
    ffv["NumValenceElectrons"]   = rdkit_vals[:, 1]
    ffv["TPSA"]                  = rdkit_vals[:, 2]

    # Reproducible 80/20 train/valid split
    ffv = ffv.sample(frac=1.0, random_state=42).reset_index(drop=True)
    n_train = int(0.8 * len(ffv))
    ffv["split"] = ["train"] * n_train + ["valid"] * (len(ffv) - n_train)
    ffv.to_csv(rdkit_path, index=False)
    print(f"Saved {rdkit_path} with {len(ffv)} rows.")
else:
    print(f"Found existing {rdkit_path} -- skipping rebuild.")

base_df = pd.read_csv(rdkit_path)
print("Train rows:", (base_df["split"] == "train").sum(),
      " Valid rows:", (base_df["split"] == "valid").sum())
base_df.head()

Found existing artifacts\ffv_features_rdkit.csv -- skipping rebuild.
Train rows: 5624  Valid rows: 1406


,id,SMILES,FFV,smiles_len,count_star,count_C,count_O,count_N,count_equal,split,MolWt,NumValenceElectrons,TPSA
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,0.374645,26,2,9,2,0,1,train,232.323,92,26.30
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,0.370410,82,2,19,0,2,0,train,598.919,236,24.06
2,388772,*Oc1ccc(S(=O)(=O)c2ccc(Oc3ccc(C4(c5ccc(Oc6ccc(...,0.378860,134,2,14,9,0,7,train,1003.207,364,122.27
3,519416,*Nc1ccc(-c2c(-c3ccc(C)cc3)c(-c3ccc(C)cc3)c(N*)...,0.387324,79,2,4,0,2,0,train,542.726,204,24.06
4,539187,*Oc1ccc(OC(=O)c2cc(OCCCCCCCCCOCC3CCCN3c3ccc([N...,0.355470,118,2,30,12,4,4,train,965.154,376,182.28


---
## CT2.1 Feature engineering (6.0 pts)

The baseline dataset already contains three RDKit descriptors: `MolWt`, `NumValenceElectrons`, and `TPSA`. Your job: **add at least 5 more descriptors of your own choice**.

How to find them: read the RDKit `Descriptors` module documentation, search online, look at papers on polymer property prediction. There are dozens of options.

For **each** descriptor you pick, write **one sentence** explaining why it might affect Fractional Free Volume (FFV) -- the fraction of empty space in a polymer. Generic statements like "this is a useful feature" do **not** earn credit. The justification should connect what the descriptor measures to a plausible structural reason for FFV going up or down.

Create a pandas DataFrame that contains the new descriptors you chose.

Save the resulting DataFrame as `artifacts/ffv_features_extended.csv` (the variable name `extended_df` is used by the sanity-check cell at the end of the notebook).

In [ ]:
extended_df = base_df.copy()

##### YOUR CODE STARTS HERE #####
# TODO: Add at least 5 more descriptors to extended_df.
#####

In [ ]:
# Save the extended features to a new CSV file in the artifacts directory.
extended_path = ARTIFACT_DIR / "ffv_features_extended.csv"
extended_df.to_csv(extended_path, index=False)
print("Saved:", extended_path)
extended_df.head()

### Reasoning

List each descriptor you picked and write **one sentence** explaining why it could plausibly affect FFV. Mention what the descriptor measures *and* the structural reason FFV would go up or down because of it.

*Your justifications here.*

---
## Train a Random Forest with your features

This part is just so you have a model whose predictions you can look at. You do not need to tune it deeply -- a default `RandomForestRegressor` is fine. 

Steps:
1. Build `X_train`, `y_train`, `X_valid`, `y_valid` from the `split` column.
2. Train `RandomForestRegressor(n_estimators=200, random_state=42)` on the training set.
3. Compute MAE, RMSE, and R² on the validation set, and print them. **The validation R² is what gets you on the leaderboard.**

In [ ]:
##### YOUR CODE STARTS HERE #####
# TODO: Create:
# - the feature matrix X_train and target vector y_train for the training split.
# - the feature matrix X_valid and target vector y_valid for the validation split.
# Use the 'split' column to separate train/valid, and remove the 'id', 'SMILES', 'FFV', and 'split' columns from the features.
#####

In [ ]:
print("Shapes:", X_train.shape, X_valid.shape)

In [ ]:
# Train a Random Forest on (X_train, y_train), predict on X_valid,
# and print MAE, RMSE, and R² on the validation set.
# TODO: your code here


---
## Feature importance and iteration

Your first 5 descriptors were a **hypothesis**. Now check what the model *actually* used.

A trained Random Forest exposes a `feature_importances_` attribute -- one score per feature, telling you how much it contributed to the predictions. Plot or print them, ranked from most to least important.

Then **iterate**: change your descriptor set and see what happens. Drop the ones the RF ignored, add new candidates you did not try the first time, use more or fewer than 5. Retrain, re-check the R², re-check the importances. The goal is to converge on the feature set that works best for predicting FFV.

In [ ]:
# Examine which features your Random Forest relied on the most.
# Use rf.feature_importances_ together with the column names of your X data,
# and plot or print them ranked from most to least important.
# TODO: your code here


### Try a different feature set

Now change your descriptors and see what improves:

- Drop the descriptors the RF barely used.
- Add at least **3 new descriptors** that you did not include in your first attempt.
- Feel free to use more (or fewer) than 5 descriptors in total.

Rebuild `extended_df`, retrain the Random Forest, and print the new validation R², MAE, RMSE together with the new feature-importance ranking. Try a couple of different combinations until you find a set you are happy with.

In [ ]:
# Pick a different set of descriptors, rebuild extended_df, retrain the Random Forest,
# and print the new R², MAE, RMSE plus the new feature-importance ranking.
# Repeat with another combination if you want.
# TODO: your code here


### Reflection

Look at the descriptors that ended up most important and compare them to the chemistry-driven hypotheses you wrote at the start.

- Were your initial guesses correct? Which descriptors did you *predict* would matter, and which *actually* did?
- Were any of the top-ranked descriptors a surprise? Try to come up with a chemistry-based explanation for why they could matter for FFV.
- Which final descriptor set are you submitting, and what is its best validation R²?

*Your reflection here (4-6 sentences).*

---
## CT2.2 Outlier investigation (4.0 pts)

Use your trained Random Forest to find the **5 polymers your model got most wrong** on the validation set. Then:

1. Compute the **absolute prediction error** for every validation polymer.
2. Pull the 5 polymers with the **largest** absolute error.
3. Draw their molecular structures with `Draw.MolsToGridImage` (look at Notebook 12 for the code to use). Include the true and predicted FFV in each legend.
4. **Write a 2-3 sentence structural hypothesis for each polymer**: what is unusual about its structure (functional groups, ring system, chain length, halogen content, charged groups, ...) that might make it hard to predict?

Vague answers like "the molecule is complex" do not earn credit. Be specific.

In [ ]:
# Identify the 5 worst-predicted polymers in the validation set, draw their
# molecular structures in a grid, and label each with its true and predicted FFV.
# TODO: your code here


### Structural hypotheses (2-3 sentences per polymer)

For each of your 5 worst-predicted polymers, fill in a hypothesis. Refer to specific structural features.

1. *Polymer 1 hypothesis...*
2. *Polymer 2 hypothesis...*
3. *Polymer 3 hypothesis...*
4. *Polymer 4 hypothesis...*
5. *Polymer 5 hypothesis...*

**Looking across all 5: do they share any common structural features?** (1-2 sentences)

*Your answer here.*

---
## Bonus (+1.0 pt) -- try one alternative model OR a small hyperparameter sweep

Pick **one** of:

- A different model: `LinearRegression`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `MLPRegressor`, or LightGBM/XGBoost if you have them installed.
- A small hyperparameter sweep: 3-4 values of `max_depth` or `n_estimators` for the RF, with a plot of validation R² vs. that parameter.

Add 2-3 sentences interpreting the result: did it help? Why or why not?

In [ ]:
# Optional bonus -- alternative model or hyperparameter sweep
# TODO: your code here


---
## Sanity checks

Run this cell after you finish CT2.1 -- this only checks that you produced the required outputs, not that they are *good*.

In [ ]:
# 1) Extended features file saved
assert (ARTIFACT_DIR / "ffv_features_extended.csv").exists(), \
    "Missing artifacts/ffv_features_extended.csv"

# 2) At least 5 new descriptor columns beyond the original set
original_cols = set(pd.read_csv(ARTIFACT_DIR / "ffv_features_rdkit.csv").columns)
new_cols = set(extended_df.columns) - original_cols
assert len(new_cols) >= 5, f"Add at least 5 new descriptor columns (you added {len(new_cols)})"
print(f"You added {len(new_cols)} new descriptors: {sorted(new_cols)}")

# 3) X / y shapes are consistent
assert X_train.ndim == 2 and X_valid.ndim == 2, "X arrays must be 2D"
assert X_train.shape[1] == X_valid.shape[1], "X_train and X_valid must have the same number of columns"
assert X_train.shape[1] >= 1, "You need at least one feature column"
assert len(y_train) == X_train.shape[0]
assert len(y_valid) == X_valid.shape[0]

print("\nBasic checks passed -- you can submit.")

---
## Submission

- Submit only this notebook (rename to `CodingTask2_<your_name>.ipynb`) by **email** (matteo.boi@unibe.ch), by **Sunday 2026-05-17, 23:59**.
- Bring your laptop to class on **2026-05-20** -- we will pool everyone's features and experiment together live.